In [1]:
import os
import platform
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (12, 6)
np.random.seed(42)

In [2]:
os.chdir(os.path.abspath(".."))
print(os.getcwd())

df = pd.read_csv("data/2025_Airbnb_NYC_listings.csv").drop(columns="Unnamed: 0")
df.head()

d:\Dev\airbnb_price_prediction


,id,source,name,description,neighborhood_overview,host_id,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,36121,city scrape,Lg Rm in Historic Prospect Heights,Cozy space share in the heart of a great neigh...,Full of tree-lined streets and beautiful brown...,62165,Michael,2009-12-11,"New York, NY",I’m an urban planner working for an internatio...,NaN,NaN,NaN,f,Prospect Heights,1.0,3.0,"['email', 'phone', 'work_email']",t,t,Neighborhood highlights,Prospect Heights,Brooklyn,40.673760,-73.966110,Private room in rental unit,Private room,1,1.0,1 shared bath,1.0,1.0,"[""Refrigerator"", ""Dishes and silverware"", ""Wif...",$200.00,90,365,90.0,90.0,365.0,365.0,90.0,365.0,NaN,t,27,57,87,362,2025-03-03,9,0,0,301,0,0,0.0,2010-12-11,2013-05-10,4.88,5.00,4.80,5.00,5.00,5.00,5.00,NaN,f,1,0,1,0,0.05
1,36647,city scrape,"1 Bedroom & your own Bathroom, Elevator Apartment",Private bedroom with your own bathroom in a 2 ...,"Manhattan, SE corner of 2nd Ave/ E. 110th street",157798,Irene,2010-07-04,"New York, NY",NaN,NaN,NaN,100%,f,East Harlem,1.0,1.0,"['email', 'phone']",t,t,Neighborhood highlights,East Harlem,Manhattan,40.792454,-73.940742,Private room in condo,Private room,2,1.0,1 private bath,1.0,1.0,"[""Oven"", ""Blender"", ""Luggage dropoff allowed"",...",$82.00,30,999,30.0,30.0,999.0,999.0,30.0,999.0,NaN,t,0,0,0,204,2025-03-03,102,0,0,143,0,0,0.0,2010-10-04,2023-12-09,4.77,4.82,4.76,4.88,4.90,4.38,4.71,NaN,f,1,0,1,0,0.58
2,38663,city scrape,Luxury Brownstone in Boerum Hill,"Beautiful, large home in great hipster neighbo...","diverse, lively, hip, cool: loaded with restau...",165789,Sarah,2010-07-13,"New York, NY",I am a lawyer and work as an executive at an a...,within a few hours,100%,40%,f,Boerum Hill,1.0,3.0,"['email', 'phone', 'work_email']",t,t,Neighborhood highlights,Boerum Hill,Brooklyn,40.684420,-73.980680,Private room in home,Private room,2,2.5,2.5 baths,5.0,5.0,"[""Portable fans"", ""Oven"", ""Baking sheet"", ""Fir...",$765.00,3,60,3.0,3.0,60.0,60.0,3.0,60.0,NaN,t,30,49,66,326,2025-03-02,43,0,0,267,0,0,0.0,2012-07-09,2023-08-30,4.70,4.83,4.52,4.88,4.88,4.86,4.62,OSE-STRREG-0001784,f,1,0,1,0,0.28
3,38833,city scrape,Spectacular West Harlem Garden Apt,This is a very large and unique space. An inc...,West Harlem is now packed with great restauran...,166532,Matthew,2010-07-14,"New York, NY",I have been a New Yorker for a long time\n and...,within an hour,100%,97%,t,Harlem,1.0,1.0,"['email', 'phone']",t,t,Neighborhood highlights,Harlem,Manhattan,40.818058,-73.946671,Entire home,Entire home/apt,2,1.0,1 bath,1.0,1.0,"[""Fire extinguisher"", ""Clothing storage: close...",$139.00,2,45,2.0,2.0,1125.0,1125.0,2.0,1125.0,NaN,t,7,18,25,25,2025-03-03,241,42,3,25,43,255,35445.0,2010-08-28,2025-02-21,4.85,4.87,4.50,4.96,4.96,4.79,4.82,OSE-STRREG-0000476,

In [3]:
df_cleaned = pd.read_csv("data/df_cleaned.csv")
df_cleaned.shape

(22308, 109)

In [19]:
df_temp = df_cleaned.copy()

In [4]:
# 날짜 컬럼 변환
df_cleaned["first_review"] = pd.to_datetime(df_cleaned["first_review"])
df_cleaned["last_review"] = pd.to_datetime(df_cleaned["last_review"])

# 파생변수: 마지막 리뷰로부터 며칠이 지났는지
df_cleaned["days_since_last_review"] = (pd.to_datetime("today") - df_cleaned["last_review"]).dt.days

In [5]:
# 날짜 컬럼 변환 2
base_date = datetime(2025, 3, 2)
df_cleaned["host_since"] = pd.to_datetime(df_cleaned["host_since"], errors="coerce")
host_since_mode = df_cleaned["host_since"].mode()[0]
df_cleaned["host_since"] = df_cleaned["host_since"].fillna(host_since_mode)
df_cleaned["host_tenure_days"] = (base_date - df_cleaned["host_since"]).dt.days
df_cleaned["host_tenure_years"] = df_cleaned["host_tenure_days"] / 365

In [ ]:
# 리뷰 및 평점 관련 수치형 컬럼 정리
review_scores = [
    "number_of_reviews",
    "reviews_per_month",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
]

df_reviews = df_cleaned[review_scores]

review_cols = [
    "number_of_reviews",
    "reviews_per_month",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
]
only_reviews = df_cleaned[review_cols]

detail_score_cols = [
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
]

detail_scores = df_cleaned[detail_score_cols]

In [6]:
# 리뷰가 하나라도 있는 숙소 (평가됨)
has_reviews = df_cleaned[df_cleaned["number_of_reviews"] > 0].copy()

# 리뷰가 아예 없는 숙소 (미평가)
no_reviews = df_cleaned[df_cleaned["number_of_reviews"] == 0].copy()

# 결과 확인
print(f"평가된 숙소: {len(has_reviews)}개")
print(f"미평가 숙소: {len(no_reviews)}개")

평가된 숙소: 15510개
미평가 숙소: 6798개


In [ ]:
# 데이터 타입, 결측치
only_reviews.info()
.info()

# 통계량 확인
display(only_reviews.describe().round(2))
display(detail_scores.describe().round(2))

# 리뷰없음(0) 제외되었는지 확인
(has_reviews.select_dtypes(include=["number"]) == 0).sum()
print(f"리뷰 있는 숙소: {len(has_reviews)}개")

<class 'pandas.DataFrame'>
RangeIndex: 22308 entries, 0 to 22307
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   number_of_reviews       22308 non-null  int64  
 1   reviews_per_month       22308 non-null  float64
 2   number_of_reviews_ltm   22308 non-null  int64  
 3   number_of_reviews_l30d  22308 non-null  int64  
 4   number_of_reviews_ly    22308 non-null  int64  
dtypes: float64(1), int64(4)
memory usage: 871.5 KB
<class 'pandas.DataFrame'>
RangeIndex: 22308 entries, 0 to 22307
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   review_scores_accuracy       22308 non-null  float64
 1   review_scores_cleanliness    22308 non-null  float64
 2   review_scores_checkin        22308 non-null  float64
 3   review_scores_communication  22308 non-null  float64
 4   review_scores_location       2

,number_of_reviews,reviews_per_month,number_of_reviews_ltm,number_of_reviews_l30d,number_of_reviews_ly
count,22308.00,22308.00,22308.00,22308.00,22308.00
mean,34.35,0.81,6.14,0.32,5.91
std,76.78,1.93,24.60,1.88,24.12
min,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,0.00
50%,5.00,0.21,0.00,0.00,0.00
75%,35.00,0.94,3.00,0.00,3.00
max,2749.00,117.98,1784.00,135.00,1797.00


,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
count,22308.00,22308.00,22308.00,22308.00,22308.00,22308.00
mean,3.30,3.26,3.36,3.35,3.29,3.21
std,2.22,2.19,2.25,2.24,2.20,2.16
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,0.00,0.00
50%,4.74,4.64,4.83,4.83,4.67,4.56
75%,4.94,4.91,4.99,5.00,4.91,4.82
max,5.00,5.00,5.00,5.00,5.00,5.00


리뷰 있는 숙소: 15510개


In [20]:
only_reviews.value_counts()

number_of_reviews  reviews_per_month  number_of_reviews_ltm  number_of_reviews_l30d  number_of_reviews_ly
0                  0.00               0                      0                       0                       6798
1                  0.05               0                      0                       0                        163
                   0.03               0                      0                       0                        134
                   0.02               0                      0                       0                        128
                   0.04               0                      0                       0                        104
                                                                                                             ... 
5                  3.85               5                      4                       0                          1
4                  2.45               4                      1                       0          

In [18]:
detail_scores.value_counts()

review_scores_accuracy  review_scores_cleanliness  review_scores_checkin  review_scores_communication  review_scores_location  review_scores_value
0.00                    0.00                       0.00                   0.00                         0.00                    0.00                   6798
5.00                    5.00                       5.00                   5.00                         5.00                    5.00                   1622
                                                                                                                               4.00                     92
                                                                                                                               4.50                     61
                        4.00                       5.00                   5.00                         5.00                    5.00                     60
                                                                              

In [ ]:
# 세부 항목 전부에 같은 평점을 준 사람들이 있다면?
same = (detail_scores.nunique(axis=1) == 1).sum()
diff = (detail_scores.nunique(axis=1) != 1).sum()

print(f"5개 컬럼 값이 모두 같은 행: {same - 6798}개")
print(f"하나라도 다른 행         : {diff}개")

5개 컬럼 값이 모두 같은 행: 1680개
하나라도 다른 행         : 13830개


In [28]:
# 모든 세부 평점이 동일한 행 추출
same_score_mask = detail_scores[detail_score_cols].nunique(axis=1) == 1

# 이 사람들이 준 점수 분포 확인
print(detail_scores[same_score_mask][detail_score_cols].iloc[:, 0].value_counts())

review_scores_accuracy
0.00    6798
5.00    1622
1.00      25
4.00      11
4.50       5
3.00       5
4.67       2
4.80       2
3.67       2
4.88       1
4.75       1
4.60       1
4.89       1
4.20       1
4.86       1
Name: count, dtype: int64


In [ ]:
# 1점 그룹 25명 확인
mask_all_same = detail_scores[detail_score_cols].nunique(axis=1) == 1
mask_one = detail_scores[detail_score_cols].iloc[:, 0] == 1

df_cleaned[mask_all_same & mask_one][
    ["description", "host_about", "license", "host_is_superhost", "price", "number_of_reviews"] + detail_score_cols
]

,description,host_about,license,host_is_superhost,price,number_of_reviews,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
4495,Feel at home wherever you choose to live with ...,"We’re Blueground, a global proptech company wi...",NaN,False,358.0,1,1.0,1.0,1.0,1.0,1.0,1.0
4681,"Comfortable , quiets rooms 20mins from Manhatt...",NaN,NaN,False,80.0,1,1.0,1.0,1.0,1.0,1.0,1.0
5778,Show up and start living from day one in Midto...,"We’re Blueground, a global proptech company wi...",NaN,False,429.0,1,1.0,1.0,1.0,1.0,1.0,1.0
6849,1 Bedroom comfortable apartment showing city v...,NaN,NaN,False,120.0,1,1.0,1.0,1.0,1.0,1.0,1.0
9516,This apartment is located in Astoria (Long Isl...,NaN,NaN,False,49.0,1,1.0,1.0,1.0,1.0,1.0,1.0
9541,"Enjoy the simplicity of this quiet, central home.",NaN,NaN,False,250.0,2,1.0,1.0,1.0,1.0,1.0,1.0
12020,Beautiful spacious studio with a large foyer a...,NaN,NaN,False,200.0,1,1.0,1.0,1.0,1.0,1.0,1.0
12756,Welcome to Furnished Habitat NYC property rent...,"Greetings and welcome, \n\nI'm Michael, the ow...",NaN,False,147.0,1,1.0,1.0,1.0,1.0,1.0,1.0
12792,Bright bedroom in a HUGE DUPLEX APARTMENT loca...,"my number: (Phone number hidden by Airbnb) , m...",NaN,False,40.0,1,1.0,1.0,1.0,1.0,1.0,1.0
12881,Enjoy a stylish experience at this centrally-l...,I’m an very friendly Brazilian guy living in N...,NaN,False,100.0,1,1.0,1.0,1.0,1.0,1.0,1.0


### 운영 방식이 이상한 숙소 정리 위해서 알아보는 중
- 세부 평점 항목들에 평점을 모두 같게 준 사람이 1680명
- 동일하게 5점을 준 사람: 1622명
- 1점:25명/4점:11명/4.5점:5명/3점:5명/4.67점:2명/4.80점:2명/3.67점:2명/4.88점:1명/4.75점:1명/4.6점:1명/4.89점:1명/4.2점:1명/4.86점:1명
- 평점이 전반적으로 상향 평준화되어 있다는 걸 eda에서 확인했음, 모두 5점을 준 사람들은 모든 항목이 만족스러웠을 가능성이 높아보임!

In [12]:
# 분석에 쓸 피처 정의
feautures = [
    # 숙소 특성
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "amenities_count",
    # 숙박 정책
    "minimum_nights",
    "instant_bookable",
    # 호스트
    "host_tenure_days",
    "host_response_rate",
    "host_acceptance_rate",
    "host_is_superhost",
    "host_listings_count",
    # 리뷰/평점 (리뷰 있는 숙소만 그대로 사용)
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "review_scores_value",
    "review_scores_checkin",
    "review_scores_communication",
    "number_of_reviews",
    # 가용성
    "availability_eoy",
]

In [ ]:
df_temp.loc[df_temp["property_type"].str.contains("hotel"), ["estimated_occupancy_l365d", "log_price"]]